# Deal Horizon Observatory — pilot walkthrough

Open this notebook from the `Capstone` folder (the same folder as the `dho_pilot` package). Run the cells from top to bottom.

The cells through the scores section stay on your machine. The last cell downloads EDGAR filings and needs a `SEC_USER_AGENT` plus network access.

## 1. Make the package importable

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "dho_pilot").is_dir():
    ROOT = Path("/Users/apizzini/Desktop/Capstone")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import dho_pilot
print("Using", ROOT)
print("Package file:", dho_pilot.__file__)


Using /Users/apizzini/Desktop/Capstone
Package file: /Users/apizzini/Desktop/Capstone/dho_pilot/__init__.py


## 2. Deal register and call budget

In [3]:
import json
from dho_pilot.__main__ import call_budget
from dho_pilot.deals import DEALS

print(json.dumps(call_budget(), indent=2))
print()
for deal in DEALS:
    print(
        f"{deal['deal_id']:16} {deal['consideration']:8} {deal['profile_status']:8} "
        f"{deal['target']} / {deal['acquirer']}"
    )


{
  "full_study_calls": 2160,
  "pilot_baseline_calls": 60,
  "pilot_probe_calls": 96,
  "pilot_total_calls": 156,
  "formula_full": "20 deals x 4 platforms x (3 prompt variants + 6 probes) x 3 reps",
  "formula_pilot": "5x4x3 baseline, then 2 deals x 4 platforms x 6 probes x 2 reps",
  "pilot_probe_ids": [
    "orientation_flip",
    "name_mask",
    "acquirer_swap",
    "premium_up",
    "persona_bear",
    "trim_regulatory"
  ]
}

CZR-FERTITTA     cash     drafted  Caesars Entertainment / Fertitta Gaming Holdco
PAYO-NUVEI       cash     drafted  Payoneer Global / Neon Maple Parent (Nuvei)
KVUE-KMB         unknown  pending  Kenvue / Kimberly-Clark
NSC-UNP          mixed    partial  Norfolk Southern / Union Pacific
UNF-CTAS         unknown  pending  UniFirst / Cintas


## 3. Market-implied close probability

Caesars is the worked example: a $31 cash offer and a 49% premium imply an unaffected price near $20.81. The $28.50 price below is illustrative. Replace it with the target's close on the forecast date before you compare it with a model.

In [4]:
import json
from dho_pilot.market_odds import annualized_spread, implied_completion, unaffected_from_premium

offer = 31.0
premium = 0.49
break_price = unaffected_from_premium(offer, premium)
price = 28.50  # illustrative; replace with the forecast-date close

odds = implied_completion(price, offer, break_price)
odds["break_price_proxy"] = round(break_price, 4)
odds["annualized_if_180_days"] = annualized_spread(odds["gross_spread"], 180)
print(json.dumps(odds, indent=2))


{
  "p_market": 0.7547728768926926,
  "gross_spread": 0.08771929824561403,
  "price_inside_anchors": true,
  "method": "two_state_blend",
  "assumptions": [
    "One close-on-terms state and one break state.",
    "Break price is an input, often the unaffected price as a proxy.",
    "Ignores ticking fees, collars, proration, and financing outs.",
    "Stock leg must use the acquirer price on the forecast date."
  ],
  "break_price_proxy": 20.8054,
  "annualized_if_180_days": 0.17787524366471733
}


## 4. Leakage scan

`dirty` is the kind of sentence that showed up in the Union Pacific draft (a vote and a schedule slip). `clean` states conditions that are still ahead. Only `dirty` should produce hits.

In [5]:
from dho_pilot.leakage import scan

dirty = (
    "Norfolk Southern stockholders voted to adopt the merger agreement. "
    "The outside date was extended after a schedule review."
)
clean = (
    "The merger agreement requires STB approval and a shareholder vote. "
    "The outside date is January 28, 2028. The regulatory termination fee is $2.5 billion."
)

print("dirty hits", len(scan(dirty)))
for hit in scan(dirty):
    print(f"  {hit['label']}: {hit['span']}")
print("clean hits", len(scan(clean)))


dirty hits 2
  vote_result: stockholders voted
  schedule_slip: outside date was extended
clean hits 0


## 5. Forecast schema

In [6]:
from dho_pilot.schemas import PROMPT_V2, validate_forecast

incomplete = {"p_close": 0.8, "confidence": 70}
complete = {
    "p_close": 0.72,
    "p_terminate": 0.18,
    "p_higher_bid": 0.11,
    "p_value_creating_36m": 0.41,
    "confidence": 64,
    "drivers": ["cash consideration", "premium", "ticking fee", "financing", "vote threshold"],
    "risks": ["regulatory delay", "financing market", "outside date"],
    "assumptions": {
        "roi_5y": {"low": 0.04, "high": 0.09},
        "wacc": {"low": 0.07, "high": 0.10},
        "roa": {"low": 0.03, "high": 0.06},
    },
    "citations": ["Merger Agreement summary"],
    "missing_information": [],
}

print("incomplete:", validate_forecast(incomplete))
print("complete:", validate_forecast(complete) or "ok")
print()
print(PROMPT_V2)


incomplete: ['missing p_terminate', 'missing p_higher_bid', 'missing p_value_creating_36m', 'drivers needs at least 5 items', 'risks needs at least 3 items', 'citations needs at least 1 items', 'assumptions must be an object with roi, wacc, roa ranges', 'missing_information must be present (use an empty list if none)']
complete: ok

You are an independent merger analyst. Use only the dossier provided. Do not use outside knowledge of later events.

Answer all four questions for every deal:
1. Probability the deal closes on the announced terms.
2. Probability the deal is terminated.
3. Probability a higher competing bid appears.
4. Probability the combination creates value for the acquirer 36 months after close.

These four probabilities are separate judgments. Do not force them to sum to 1.

Return JSON only with this shape:
{
  "p_close": 0.0,
  "p_terminate": 0.0,
  "p_higher_bid": 0.0,
  "p_value_creating_36m": 0.0,
  "confidence": 0,
  "drivers": ["", "", "", "", ""],
  "risks": [""

## 6. Forecast ledger

This writes two demo forecasts into a temporary file, checks the chain, then edits the first probability so you can see verification fail.

In [7]:
import json
import tempfile
from pathlib import Path
from dho_pilot.ledger import append, verify

path = Path(tempfile.mkdtemp()) / "ledger.jsonl"

first = append(path, {"deal_id": "CZR-FERTITTA", "platform": "demo", "p_close": 0.72})
second = append(path, {"deal_id": "NSC-UNP", "platform": "demo", "p_close": 0.55})
print("intact:", verify(path))
print("second entry points at first hash:", second["prev_hash"] == first["entry_hash"])

lines = path.read_text().splitlines()
forged = json.loads(lines[0])
forged["payload"]["p_close"] = 0.99
lines[0] = json.dumps(forged)
path.write_text("\n".join(lines) + "\n")
print("after edit:", verify(path))


intact: (True, '2 entries verified; head 0d7194b12761')
second entry points at first hash: True
after edit: (False, 'hash mismatch at entry 0')


## 7. Scores

The sensitivity example uses made-up deltas so you can see the index. Brier and log loss need a real 0/1 outcome; here the imagined outcome is that the deal closed.

In [8]:
from dho_pilot.deals import HYPOTHESES
from dho_pilot.score import brier, counterfactual_sensitivity, log_loss

observations = [
    {"variable": "premium", "delta_p_close": 0.06, "expected_sign": 1},
    {"variable": "regulatory_gravity", "delta_p_close": -0.08, "expected_sign": -1},
    {"variable": "closing_horizon", "delta_p_close": 0.03, "expected_sign": -1},
]
csi = counterfactual_sensitivity(observations)
print(f"CSI {csi['csi']:.3f} on {csi['n']} checks")
for row in csi["rows"]:
    print(f"  {row['variable']:24} delta {row['delta_p_close']:+.2f}  agree {row['agree']}")
print("Brier if the deal closes and the model said 0.72:", round(brier(0.72, 1), 4))
print("Log loss for the same call:", round(log_loss(0.72, 1), 4))
print()
for item in HYPOTHESES:
    print(f"{item['variable']:24} expected sign {item['expected_sign_p_close']:+d}")


CSI 0.667 on 3 checks
  premium                  delta +0.06  agree True
  regulatory_gravity       delta -0.08  agree True
  closing_horizon          delta +0.03  agree False
Brier if the deal closes and the model said 0.72: 0.0784
Log loss for the same call: 0.3285

premium                  expected sign +1
regulatory_gravity       expected sign -1
reverse_termination_fee  expected sign +1
cash_share               expected sign +1
closing_horizon          expected sign -1
synergy_to_ebitda        expected sign +0


## 8. Download filings (optional)

Uncomment `SEC_USER_AGENT` and put in your name and email. The SEC asks requesters to identify themselves and to stay under their published rate limit. This cell downloads every filing in the register into `dho_data/raw/`.

In [9]:
import os
from dho_pilot.fetch_filings import fetch_pilot

# os.environ["SEC_USER_AGENT"] = "Anne Pizzini MSDS498 you@email.com"

if not os.environ.get("SEC_USER_AGENT"):
    print("Set SEC_USER_AGENT in the line above, then run this cell again.")
else:
    records = fetch_pilot(ROOT / "dho_data")
    for record in records:
        coverage = record.get("coverage") or {}
        missing = ", ".join(coverage.get("missing") or []) or "-"
        print(
            f"{record.get('deal_id'):16} {str(record.get('form')):8} "
            f"ok={record.get('ok')} bytes={record.get('bytes')} "
            f"coverage={coverage.get('coverage')} missing={missing}"
        )


Set SEC_USER_AGENT in the line above, then run this cell again.


## 9. Run the model comparison

The keys live in `secrets.env` in the Capstone folder. This cell loads that file inside `main` and does not print the keys.

Leave `--dry-run` in place for a cost estimate. Delete `"--dry-run"` from the list when you want the models to answer. That call spends API credits. xAI is skipped while its key is blank.

In [ ]:
from run_llm_comparison import main

main(["--deal", "CZR-FERTITTA", "--dry-run"])
